# GRACE: Tái Tạo Pipeline Đúng Theo Bài Báo

Notebook này implement chính xác pipeline GRACE như được mô tả trong bài báo:
- **Module A:** Demonstration Selection (CodeT5 → **T-SNE** → **L₂ distance** → Jaccard + **SimSBT** Levenshtein)
- **Module B:** Graph Structure Information Generation (**Joern** CPG)
- **Module C:** Enhanced Vulnerability Detection (**Vertex AI Qwen3-Coder**)

### Sai khác chính so với code gốc:
| Khía cạnh | Code Gốc | Paper (Notebook này) |
|-----------|----------|----------------------|
| Giảm chiều | BERT Whitening | **T-SNE** |
| Tìm kiếm ngữ nghĩa | FAISS Inner Product | **L₂ distance** |
| Tương đồng cú pháp | Levenshtein trên AST thô | **SimSBT + Levenshtein** |
| Tỷ lệ split | Tùy ý | **8:1:1** |
| Label của example | Không gửi | **Gửi kèm label** |

### Môi trường chạy:
- WSL với venv tại `/mnt/c/MY-FILES/LAB-THAY-THO/Grace-Code-Based/venv`
- Thư mục làm việc: `/mnt/c/MY-FILES/LAB-THAY-THO/Grace-Paper-Based`

## Bước 1: Cài đặt thư viện

Thêm `scikit-learn` (chứa T-SNE) so với notebook code-based.

In [12]:
%pip install torch transformers faiss-cpu python-Levenshtein scikit-learn scipy pandas tqdm gdown google-cloud-aiplatform

Note: you may need to restart the kernel to use updated packages.


## Bước 2: Tải dataset Devign (FFmpeg+Qemu)

Paper sử dụng 3 dataset: Devign (FFmpeg+Qemu), Big-Vul, Reveal.
Notebook này chạy trên **Devign** (27,318 hàm C/C++).

In [13]:
from datasets import load_dataset
print('Loading claudios/ReVeal dataset...')
ds = load_dataset('claudios/ReVeal')

# Reveal đã chia sẵn train/test
train_code_list = ds['train']['functionSource']
train_target_list = ds['train']['label']
test_code_list = ds['test']['functionSource']
test_target_list = ds['test']['label']

NUM_TRAIN = len(train_code_list)
NUM_TEST = len(test_code_list)

print(f'Train size: {NUM_TRAIN}')
print(f'Test size: {NUM_TEST}')


Loading claudios/ReVeal dataset...
Train size: 18187
Test size: 2274


## Bước 3: Chia Train/Val/Test = 8:1:1 (đúng paper)

Paper: *"We split the dataset into train, validation, and test sets with a ratio of 8:1:1."*

Sử dụng stratified split để giữ tỷ lệ vulnerable/non-vulnerable cân bằng.

In [14]:
import os
import json
import random
import numpy as np
# Dataset Reveal đã được chia sẵn train/test từ HuggingFace


## Bước 4: Trích xuất CPG + SimSBT bằng Joern (16 cores)

Sử dụng Joern để tạo Code Property Graph (AST + PDG + CFG).

**Khác biệt so với code gốc:** Ngoài node_str và edge_str, còn tạo thêm chuỗi **SimSBT**
(Structure-Based Traversal trên type nodes) để dùng cho bước tính syntactic similarity.

Paper: *"We use SimSBT to traverse AST type nodes to generate ordered sequences."*

In [15]:
import os
import subprocess
import shutil
import pandas as pd
from tqdm import tqdm
import concurrent.futures

# === Giải nén Joern nếu chưa có ===
joern_dir = '../Joern/joern'
if not os.path.exists(joern_dir):
    import zipfile
    print('Đang giải nén Joern...')
    with zipfile.ZipFile('../Joern/joern.zip', 'r') as z:
        z.extractall('../Joern/')
    print('[OK] Đã giải nén Joern')
else:
    print('[OK] Joern đã sẵn sàng')

JOERN_PARSE = '../Joern/joern/joern/joern/joern-parse'
os.chmod(JOERN_PARSE, 0o755)

def build_simsbt(nodes_csv_path, edges_csv_path):
    """
    SimSBT: Structure-Based Traversal trên type nodes của AST.
    Paper: 'traverse AST type nodes to generate ordered sequences'
    Output: '(FunctionDef (ParameterList (Parameter )Parameter )ParameterList )FunctionDef'
    """
    try:
        nodes_df = pd.read_csv(nodes_csv_path, sep='\t', on_bad_lines='skip', engine='python').fillna('')
        edges_df = pd.read_csv(edges_csv_path, sep='\t', on_bad_lines='skip', engine='python').fillna('')
    except Exception:
        return ''

    # Build node dict: key -> type
    node_types = {}
    for _, row in nodes_df.iterrows():
        node_types[str(row.get('key', ''))] = str(row.get('type', ''))

    # Build tree from IS_AST_PARENT edges
    # Old Joern: start IS_AST_PARENT end => end is parent of start
    children = {}  # parent -> [children]
    has_parent = set()
    ast_nodes = set()

    for _, row in edges_df.iterrows():
        if str(row.get('type', '')) == 'IS_AST_PARENT':
            child = str(row.get('start', ''))
            parent = str(row.get('end', ''))
            children.setdefault(parent, []).append(child)
            has_parent.add(child)
            ast_nodes.update([child, parent])

    if not ast_nodes:
        return ''

    # Tìm root (node không có parent)
    roots = ast_nodes - has_parent
    if not roots:
        return ''
    root = sorted(roots, key=lambda x: int(x) if x.isdigit() else 0)[0]

    # DFS SimSBT traversal (chỉ output type names)
    def dfs(node_key, depth=0):
        if depth > 100:
            return ''
        ntype = node_types.get(node_key, 'Unknown')
        result = f'({ntype} '
        for child_key in sorted(children.get(node_key, []), key=lambda x: int(x) if x.isdigit() else 0):
            result += dfs(child_key, depth + 1)
        result += f'){ntype} '
        return result

    return dfs(root).strip()


def extract_graph_worker(args):
    """Worker function cho multiprocessing. Trả về (idx, node_str, edge_str, sbt_str)."""
    func_code, idx = args
    tmp_dir = f'_joern_tmp_{idx}'
    out_dir = f'_joern_out_{idx}'

    for d in [tmp_dir, out_dir]:
        if os.path.exists(d):
            shutil.rmtree(d, ignore_errors=True)
    os.makedirs(tmp_dir, exist_ok=True)

    func_file = os.path.join(tmp_dir, f'func_{idx}.c')
    with open(func_file, 'w', encoding='utf-8') as f:
        f.write(func_code)

    try:
        subprocess.run([JOERN_PARSE, tmp_dir, out_dir],
                       capture_output=True, text=True, timeout=15)
    except Exception:
        for d in [tmp_dir, out_dir]:
            if os.path.exists(d):
                shutil.rmtree(d, ignore_errors=True)
        return idx, '', '', ''

    node_csv = os.path.join(out_dir, tmp_dir, f'func_{idx}.c', 'nodes.csv')
    edge_csv = os.path.join(out_dir, tmp_dir, f'func_{idx}.c', 'edges.csv')

    node_str, edge_str, sbt_str = '', '', ''

    if os.path.exists(node_csv):
        try:
            df = pd.read_csv(node_csv, sep='\t', on_bad_lines='skip', engine='python').fillna('')
            parts = []
            for _, row in df.iterrows():
                ntype = str(row.get('type', ''))
                ncode = str(row.get('code', '')).replace('\n', ' ').strip()
                parts.append(f'{ntype}({ncode})' if ncode else ntype)
            node_str = ' '.join(parts)
        except Exception:
            pass

    if os.path.exists(edge_csv):
        try:
            df = pd.read_csv(edge_csv, sep='\t', on_bad_lines='skip', engine='python').fillna('')
            parts = []
            for _, row in df.iterrows():
                s = str(row.get('start', ''))
                e = str(row.get('end', ''))
                t = str(row.get('type', ''))
                parts.append(f'{s}->{e}[{t}]')
            edge_str = ' '.join(parts)
        except Exception:
            pass

    # Build SimSBT
    if os.path.exists(node_csv) and os.path.exists(edge_csv):
        sbt_str = build_simsbt(node_csv, edge_csv)

    # Dọn dẹp
    for d in [tmp_dir, out_dir]:
        if os.path.exists(d):
            shutil.rmtree(d, ignore_errors=True)

    return idx, node_str, edge_str, sbt_str


def run_extraction_parallel(code_list, start_idx=0, max_workers=16):
    results = [None] * len(code_list)
    tasks = [(code, start_idx + i) for i, code in enumerate(code_list)]

    with concurrent.futures.ProcessPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(extract_graph_worker, task): i for i, task in enumerate(tasks)}
        for future in tqdm(concurrent.futures.as_completed(futures), total=len(tasks)):
            list_idx = futures[future]
            try:
                _, node_str, edge_str, sbt_str = future.result()
                results[list_idx] = (node_str, edge_str, sbt_str)
            except Exception:
                results[list_idx] = ('', '', '')

    return (
        [r[0] for r in results],  # node_list
        [r[1] for r in results],  # edge_list
        [r[2] for r in results],  # sbt_list
    )


import pickle

# === Trích xuất (có cache) ===
cache_file = 'joern_cache.pkl'
if os.path.exists(cache_file):
    print(f"Đang tải kết quả Joern từ cache: {cache_file}")
    with open(cache_file, 'rb') as f:
        cache_data = pickle.load(f)
        train_node_list = cache_data['train_node_list']
        train_edge_list = cache_data['train_edge_list']
        train_sbt_list = cache_data['train_sbt_list']
        test_node_list = cache_data['test_node_list']
        test_edge_list = cache_data['test_edge_list']
        test_sbt_list = cache_data['test_sbt_list']
else:
    print('=== Trích xuất CPG + SimSBT cho tập TRAIN (16 cores) ===')
    train_node_list, train_edge_list, train_sbt_list = run_extraction_parallel(
        train_code_list, start_idx=0, max_workers=16)
    
    print('\n=== Trích xuất CPG + SimSBT cho tập TEST (16 cores) ===')
    test_node_list, test_edge_list, test_sbt_list = run_extraction_parallel(
        test_code_list, start_idx=NUM_TRAIN, max_workers=16)
    
    print('\nLưu kết quả vào cache...')
    with open(cache_file, 'wb') as f:
        pickle.dump({
            'train_node_list': train_node_list,
            'train_edge_list': train_edge_list,
            'train_sbt_list': train_sbt_list,
            'test_node_list': test_node_list,
            'test_edge_list': test_edge_list,
            'test_sbt_list': test_sbt_list,
        }, f)
print(f'\n[OK] Train: {sum(1 for n in train_node_list if n)}/{NUM_TRAIN} hàm có đồ thị')
print(f'[OK] Test: {sum(1 for n in test_node_list if n)}/{NUM_TEST} hàm có đồ thị')
print(f'[OK] Train SBT: {sum(1 for s in train_sbt_list if s)}/{NUM_TRAIN} hàm có SimSBT')


[OK] Joern đã sẵn sàng
=== Trích xuất CPG + SimSBT cho tập TRAIN (16 cores) ===


100%|██████████| 18187/18187 [00:10<00:00, 1665.52it/s]


=== Trích xuất CPG + SimSBT cho tập TEST (16 cores) ===



100%|██████████| 2274/2274 [00:02<00:00, 1106.44it/s]


Lưu kết quả vào cache...

[OK] Train: 0/18187 hàm có đồ thị
[OK] Test: 0/2274 hàm có đồ thị
[OK] Train SBT: 0/18187 hàm có SimSBT


## Bước 5: Mã hoá mã nguồn bằng CodeT5

Dùng CodeT5 (Salesforce/codet5-base) để encode toàn bộ train + test code
thành vectors 768 chiều. Đây là bước đầu tiên của Module A (Demonstration Selection).

In [16]:
import torch
from transformers import RobertaTokenizer, RobertaModel

# === Tải CodeT5 (workaround cho lỗi tokenizer_config trên transformers mới) ===
import urllib.request
if not os.path.exists('vocab.json'):
    urllib.request.urlretrieve(
        'https://huggingface.co/Salesforce/codet5-base/resolve/main/vocab.json', 'vocab.json')
if not os.path.exists('merges.txt'):
    urllib.request.urlretrieve(
        'https://huggingface.co/Salesforce/codet5-base/resolve/main/merges.txt', 'merges.txt')

tokenizer = RobertaTokenizer(vocab_file='vocab.json', merges_file='merges.txt')
codet5_model = RobertaModel.from_pretrained('Salesforce/codet5-base')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
codet5_model.to(DEVICE)
codet5_model.eval()
print(f'[OK] CodeT5 loaded on {DEVICE}')

def encode_batch(code_list, batch_size=32):
    """Encode danh sách code thành vectors bằng CodeT5 (CLS token)."""
    all_vecs = []
    for i in tqdm(range(0, len(code_list), batch_size), desc='Encoding'):
        batch = code_list[i:i + batch_size]
        inputs = tokenizer(
            batch, padding=True, truncation=True, max_length=512, return_tensors='pt'
        ).to(DEVICE)
        with torch.no_grad():
            outputs = codet5_model(**inputs)
        vecs = outputs.last_hidden_state[:, 0, :].cpu().numpy()  # CLS token
        all_vecs.append(vecs)
    return np.vstack(all_vecs)

cache_file_codet5 = 'codet5_cache.npz'
if os.path.exists(cache_file_codet5):
    print(f"Đang tải vectors từ cache: {cache_file_codet5}")
    data = np.load(cache_file_codet5)
    train_vecs = data['train_vecs']
    test_vecs = data['test_vecs']
else:
    print('Encoding train code...')
    train_vecs = encode_batch(train_code_list)
    print('Encoding test code...')
    test_vecs = encode_batch(test_code_list)
    print('Lưu vectors vào cache...')
    np.savez(cache_file_codet5, train_vecs=train_vecs, test_vecs=test_vecs)
print(f'Train vectors: {train_vecs.shape}')
print(f'Test vectors: {test_vecs.shape}')

[transformers] You are using a model of type `t5` to instantiate a model of type `roberta`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.
Loading weights: 0it [00:00, ?it/s]
[transformers] RobertaModel LOAD REPORT from: Salesforce/codet5-base
Key                                                                  | Status     | 
---------------------------------------------------------------------+------------+-
encoder.block.{0...11}.layer.{0, 1}.layer_norm.weight                | UNEXPECTED | 
decoder.block.{0...11}.layer.2.DenseReluDense.wi.weight              | UNEXPECTED | 
decoder.block.{0...11}.layer.0.SelfAttention.o.weight                | UNEXPECTED | 
decoder.block.{0...11}.layer.1.EncDecAttention.o.weight              | UNEXPECTED | 


[OK] CodeT5 loaded on cpu
Encoding train code...


Encoding: 100%|██████████| 569/569 [01:09<00:00,  8.16it/s]


Encoding test code...


Encoding: 100%|██████████| 72/72 [00:09<00:00,  7.27it/s]


Lưu vectors vào cache...
Train vectors: (18187, 768)
Test vectors: (2274, 768)


## Bước 6: T-SNE + L₂ Distance (Demonstration Selection — Phần 1)

**KHÁC BIỆT LỚN SO VỚI CODE GỐC:**
- Code gốc: BERT Whitening (768→256) + FAISS Inner Product
- Paper: **T-SNE** giảm chiều + **L₂ (Euclidean) distance**

Paper: *"We apply T-SNE to reduce the dimensionality of the feature embeddings,
then compute L₂ distance to measure semantic similarity."*

In [17]:
from sklearn.manifold import TSNE
from scipy.spatial.distance import cdist

TOP_K = 5  # Paper không chỉ rõ, dùng giá trị từ code gốc

# T-SNE giảm chiều (đúng paper)
print('Đang chạy T-SNE trên toàn bộ vectors (có thể mất 5-15 phút)...')
all_vecs = np.vstack([train_vecs, test_vecs])
tsne = TSNE(
    n_components=2,        # Chuẩn T-SNE
    method='barnes_hut',   # O(N log N) thay vì O(N²)
    random_state=42,
    perplexity=30,
    max_iter=1000,
    verbose=1
)
all_reduced = tsne.fit_transform(all_vecs)

train_reduced = all_reduced[:NUM_TRAIN]
test_reduced = all_reduced[NUM_TRAIN:]
print(f'T-SNE output shape: {all_reduced.shape}')

# Tính L₂ distance (đúng paper)
print('Tính L₂ distance giữa test và train...')
distances = cdist(test_reduced, train_reduced, metric='euclidean')

# Lấy top-K candidates gần nhất
topk_indices = np.argsort(distances, axis=1)[:, :TOP_K]
print(f'[OK] Đã tìm top-{TOP_K} candidates cho {NUM_TEST} test functions')

Đang chạy T-SNE trên toàn bộ vectors (có thể mất 5-15 phút)...
[t-SNE] Computing 91 nearest neighbors...
[t-SNE] Indexed 20461 samples in 0.002s...
[t-SNE] Computed neighbors for 20461 samples in 3.530s...
[t-SNE] Computed conditional probabilities for sample 1000 / 20461
[t-SNE] Computed conditional probabilities for sample 2000 / 20461
[t-SNE] Computed conditional probabilities for sample 3000 / 20461
[t-SNE] Computed conditional probabilities for sample 4000 / 20461
[t-SNE] Computed conditional probabilities for sample 5000 / 20461
[t-SNE] Computed conditional probabilities for sample 6000 / 20461
[t-SNE] Computed conditional probabilities for sample 7000 / 20461
[t-SNE] Computed conditional probabilities for sample 8000 / 20461
[t-SNE] Computed conditional probabilities for sample 9000 / 20461
[t-SNE] Computed conditional probabilities for sample 10000 / 20461
[t-SNE] Computed conditional probabilities for sample 11000 / 20461
[t-SNE] Computed conditional probabilities for sample 1

## Bước 7: Tính Mixed Score (Demonstration Selection — Phần 2)

**KHÁC BIỆT SO VỚI CODE GỐC:**
- Code gốc: Levenshtein trên chuỗi AST thô
- Paper: Levenshtein trên chuỗi **SimSBT** (type nodes)

Paper formulas:
- `lexical_similarity(A, B) = |Ω ∩ Γ| / |Ω ∪ Γ|` (Jaccard)
- `syntactic_similarity(A, B) = (len(a) + len(b) - lev) / (len(a) + len(b))` (SimSBT + Levenshtein)
- `mixed_score = ω × lexical + (1-ω) × syntactic`

In [18]:
import Levenshtein

OMEGA = 0.7  # Paper không chỉ rõ, dùng giá trị từ code gốc

def jaccard_similarity(s1, s2):
    """Jaccard similarity trên token sets (lexical similarity). Đúng paper."""
    set1, set2 = set(s1.split()), set(s2.split())
    union = set1 | set2
    if not union:
        return 0.0
    return len(set1 & set2) / len(union)

def syntactic_similarity_simsbt(sbt1, sbt2):
    """
    Syntactic similarity = (len(a) + len(b) - lev) / (len(a) + len(b))
    Dùng SimSBT sequences (đúng paper) thay vì AST thô (code gốc).
    """
    seq1 = sbt1.split()
    seq2 = sbt2.split()
    if not seq1 and not seq2:
        return 0.0
    return Levenshtein.seqratio(seq1, seq2)

# Tìm best demonstration cho mỗi test function
print('Tính mixed score và chọn demonstration...')
best_examples = []
for i in tqdm(range(NUM_TEST)):
    best_score = -1
    best_idx = topk_indices[i][0]

    for j in topk_indices[i]:
        lex_sim = jaccard_similarity(test_code_list[i], train_code_list[j])
        syn_sim = syntactic_similarity_simsbt(
            test_sbt_list[i] if test_sbt_list[i] else '',
            train_sbt_list[j] if train_sbt_list[j] else ''
        )
        mixed = OMEGA * lex_sim + (1 - OMEGA) * syn_sim

        if mixed > best_score:
            best_score = mixed
            best_idx = j

    best_examples.append({
        'code': train_code_list[best_idx],
        'label': 'Vulnerable' if train_target_list[best_idx] == 1 else 'Non-vulnerable',
        'score': best_score
    })

print(f'[OK] Đã chọn demonstration cho {len(best_examples)} test functions')
print(f'Average mixed score: {np.mean([e["score"] for e in best_examples]):.4f}')

Tính mixed score và chọn demonstration...


100%|██████████| 2274/2274 [00:04<00:00, 565.19it/s]

[OK] Đã chọn demonstration cho 2274 test functions
Average mixed score: 0.1449


## Bước 8: Ghép dữ liệu vào JSON

Tạo `devign_test_processed.json` với các trường:
- `func`: mã nguồn hàm
- `target`: nhãn thực (0 hoặc 1)
- `node`: thông tin node từ CPG
- `edge`: thông tin edge từ CPG
- `example`: mã nguồn demonstration (hàm tương tự nhất)
- `example_label`: nhãn của demonstration ('Vulnerable' hoặc 'Non-vulnerable')

In [19]:
output_data = []
for i in range(NUM_TEST):
    output_data.append({
        'func': test_code_list[i],
        'target': test_target_list[i],
        'node': test_node_list[i],
        'edge': test_edge_list[i],
        'example': best_examples[i]['code'],
        'example_label': best_examples[i]['label']
    })

with open('reveal_test_processed.json', 'w') as f:
    json.dump(output_data, f)

print(f'[OK] Đã tạo reveal_test_processed.json với {len(output_data)} hàm')
print(f'Vuln: {sum(1 for d in output_data if d["target"] == 1)}')


[OK] Đã tạo reveal_test_processed.json với 2274 hàm
Vuln: 230


## Bước 9: Kiểm tra kết nối Vertex AI

Paper dùng GPT-4. Ta thay bằng **Qwen3-Coder-480B** qua Vertex AI.
Đây là giới hạn duy nhất so với paper (do không có API key GPT-4).

In [20]:
import vertexai
from vertexai.generative_models import GenerativeModel

vertexai.init(project='grace-enhanced', location='global')
llm_model = GenerativeModel('publishers/qwen/models/qwen3-coder-480b-a35b-instruct-maas')

try:
    response = llm_model.generate_content('Say hello!')
    print(f'Response: {response.text}')
    print('[OK] Vertex AI đã sẵn sàng')
except Exception as e:
    print(f'[LỖI] {e}')
    print('Hãy kiểm tra lại project ID và quyền truy cập.')

/media/quang-dung/Windows-SSD/MY-FILES/LAB-THAY-THO/.venv-while-ubuntu/lib/python3.12/site-packages/vertexai/generative_models/_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()
/media/quang-dung/Windows-SSD/MY-FILES/LAB-THAY-THO/.venv-while-ubuntu/lib/python3.12/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


Response: Hello! It's nice to meet you! How can I help you today?
[OK] Vertex AI đã sẵn sàng


## Bước 10: Ghi đè basep.py và llmpre.py

Ghi đè hoàn toàn 2 file này với phiên bản:
- Dùng **Vertex AI** thay cho OpenAI
- Prompt **đúng paper**: output 'Vulnerable' hoặc 'Non-vulnerable'
- Logic trích xuất kết quả dùng **keyword matching** (robust)
- `llmpre.py` gửi kèm **label của demonstration** (đúng paper)

In [21]:
# === Ghi đè basep.py ===
basep_code = 'import vertexai\nfrom vertexai.generative_models import GenerativeModel\nimport json\nimport csv\nimport logging\n\n# === Vertex AI (thay cho GPT-4 trong paper) ===\nvertexai.init(project="grace-enhanced", location="global")\nmodel = GenerativeModel("publishers/qwen/models/qwen3-coder-480b-a35b-instruct-maas")\n\n# === Prompt đúng paper ===\ntemplates = {\n    1: ("In the above code snippet, check for potential security vulnerabilities "\n        "and output either \'Vulnerable\' or \'Non-vulnerable\'. "\n        "You are now an excellent programmer."\n        "You are conducting a function vulnerability detection task for C/C++ language."),\n}\n\nlogger = logging.getLogger(__name__)\nlogger.setLevel(logging.INFO)\nfh = logging.FileHandler(\'revealmetrics_basep.log\')\nfh.setFormatter(logging.Formatter(\'%(asctime)s - %(name)s - %(levelname)s - %(message)s\'))\nlogger.addHandler(fh)\n\n\ndef extract_prediction(text):\n    """Trích xuất kết quả bằng keyword matching (robust hơn exact match)."""\n    text_lower = text.strip().lower()\n    if \'non-vulnerable\' in text_lower or \'non vulnerable\' in text_lower:\n        return 0\n    elif \'vulnerable\' in text_lower:\n        return 1\n    # Fallback: thử tìm số 0 hoặc 1\n    text_clean = text.strip().replace(\'`\', \'\').strip()\n    if text_clean == \'0\':\n        return 0\n    elif text_clean == \'1\':\n        return 1\n    return 2\n\n\ndef main():\n    with open(\'reveal_test_processed.json\', \'r\') as f:\n        data = json.load(f)\n\n    def calculate_metrics(preds, truths):\n        tp = sum(1 for p, t in zip(preds, truths) if p == t == 1)\n        tn = sum(1 for p, t in zip(preds, truths) if p == t == 0)\n        fp = sum(1 for p, t in zip(preds, truths) if p == 1 and t == 0)\n        fn = sum(1 for p, t in zip(preds, truths) if p == 0 and t == 1)\n        n = len(preds)\n        acc = (tp + tn) / n if n else 0\n        prec = tp / (tp + fp) if (tp + fp) else 0\n        rec = tp / (tp + fn) if (tp + fn) else 0\n        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0\n        return acc, prec, rec, f1\n\n    prediction_ls = []\n    ground_truth = []\n\n    for row in data:\n        inputCode = row[\'func\'][:4000]\n        prompt = inputCode + templates[1]\n\n        try:\n            response = model.generate_content(prompt)\n            raw = response.text.strip()\n        except Exception as e:\n            logger.error(f"Error: {e}")\n            raw = ""\n\n        prediction = extract_prediction(raw)\n        print(f"Raw: {raw[:80]}... => {prediction}")\n\n        prediction_ls.append(prediction)\n        ground_truth.append(row[\'target\'])\n\n        # Lưu tiến trình\n        with open(\'revealresults_basep.csv\', \'w\', newline=\'\') as f:\n            writer = csv.writer(f)\n            writer.writerow([\'Prediction\', \'Groundtruth\'])\n            writer.writerows(zip(prediction_ls, ground_truth))\n\n        acc, prec, rec, f1 = calculate_metrics(prediction_ls, ground_truth)\n        msg = f"[{len(prediction_ls)}/{len(data)}] Acc:{acc:.4f} P:{prec:.4f} R:{rec:.4f} F1:{f1:.4f}"\n        print(msg)\n        logger.info(msg)\n\n\nif __name__ == \'__main__\':\n    main()\n'

with open('basep.py', 'w', encoding='utf-8') as f:
    f.write(basep_code)
print('[OK] Đã ghi đè basep.py')

# === Ghi đè llmpre.py ===
llmpre_code = 'import vertexai\nfrom vertexai.generative_models import GenerativeModel\nimport json\nimport csv\nimport logging\n\n# === Vertex AI (thay cho GPT-4 trong paper) ===\nvertexai.init(project="grace-enhanced", location="global")\nmodel = GenerativeModel("publishers/qwen/models/qwen3-coder-480b-a35b-instruct-maas")\n\n# === Prompt đúng paper ===\ntemplates = {\n    1: ("In the above code snippet, check for potential security vulnerabilities "\n        "and output either \'Vulnerable\' or \'Non-vulnerable\'. "\n        "You are now an excellent programmer."\n        "You are conducting a function vulnerability detection task for C/C++ language."),\n    2: "The node information of the function is as follows:",\n    3: "The edge information of the function is as follows:",\n    4: "Here is an example for you to learn from:",\n}\n\nlogger = logging.getLogger(__name__)\nlogger.setLevel(logging.INFO)\nfh = logging.FileHandler(\'revealmetrics_llmpre.log\')\nfh.setFormatter(logging.Formatter(\'%(asctime)s - %(name)s - %(levelname)s - %(message)s\'))\nlogger.addHandler(fh)\n\n\ndef extract_prediction(text):\n    """Trích xuất kết quả bằng keyword matching (robust hơn exact match)."""\n    text_lower = text.strip().lower()\n    if \'non-vulnerable\' in text_lower or \'non vulnerable\' in text_lower:\n        return 0\n    elif \'vulnerable\' in text_lower:\n        return 1\n    text_clean = text.strip().replace(\'`\', \'\').strip()\n    if text_clean == \'0\':\n        return 0\n    elif text_clean == \'1\':\n        return 1\n    return 2\n\n\ndef main():\n    with open(\'reveal_test_processed.json\', \'r\') as f:\n        data = json.load(f)\n\n    def calculate_metrics(preds, truths):\n        tp = sum(1 for p, t in zip(preds, truths) if p == t == 1)\n        tn = sum(1 for p, t in zip(preds, truths) if p == t == 0)\n        fp = sum(1 for p, t in zip(preds, truths) if p == 1 and t == 0)\n        fn = sum(1 for p, t in zip(preds, truths) if p == 0 and t == 1)\n        n = len(preds)\n        acc = (tp + tn) / n if n else 0\n        prec = tp / (tp + fp) if (tp + fp) else 0\n        rec = tp / (tp + fn) if (tp + fn) else 0\n        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0\n        return acc, prec, rec, f1\n\n    prediction_ls = []\n    ground_truth = []\n\n    for row in data:\n        inputCode = row[\'func\'][:4000]\n        inputnode = row.get(\'node\', \'\')[:2000]\n        inputedge = row.get(\'edge\', \'\')[:2000]\n        inputex = row.get(\'example\', \'\')[:4000]\n        example_label = row.get(\'example_label\', \'\')\n\n        # Prompt đúng paper: code + task + identity + domain + graph + example + label\n        prompt = (\n            inputCode + templates[1]\n            + templates[2] + inputnode\n            + templates[3] + inputedge\n            + templates[4] + inputex\n            + f"\\nThis example is {example_label}."\n        )\n\n        try:\n            response = model.generate_content(prompt)\n            raw = response.text.strip()\n        except Exception as e:\n            logger.error(f"Error: {e}")\n            raw = ""\n\n        prediction = extract_prediction(raw)\n        print(f"Raw: {raw[:80]}... => {prediction}")\n\n        prediction_ls.append(prediction)\n        ground_truth.append(row[\'target\'])\n\n        # Lưu tiến trình\n        with open(\'revealresults_llmpre.csv\', \'w\', newline=\'\') as f:\n            writer = csv.writer(f)\n            writer.writerow([\'Prediction\', \'Groundtruth\'])\n            writer.writerows(zip(prediction_ls, ground_truth))\n\n        acc, prec, rec, f1 = calculate_metrics(prediction_ls, ground_truth)\n        msg = f"[{len(prediction_ls)}/{len(data)}] Acc:{acc:.4f} P:{prec:.4f} R:{rec:.4f} F1:{f1:.4f}"\n        print(msg)\n        logger.info(msg)\n\n\nif __name__ == \'__main__\':\n    main()\n'

with open('llmpre.py', 'w', encoding='utf-8') as f:
    f.write(llmpre_code)


[OK] Đã ghi đè basep.py


## Bước 11: Chạy Baseline (basep.py)

Chỉ gửi mã nguồn + prompt cơ bản (không có graph, không có example).
Đây là baseline để so sánh với GRACE.

In [22]:
!python3 basep.py

/media/quang-dung/Windows-SSD/MY-FILES/LAB-THAY-THO/.venv-while-ubuntu/lib/python3.12/site-packages/vertexai/generative_models/_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()
/media/quang-dung/Windows-SSD/MY-FILES/LAB-THAY-THO/.venv-while-ubuntu/lib/python3.12/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)
Raw: Looking at this code snippet, I need to analyze it for potential security vulner... => 1
[1/2274] Acc:0.0000 P:

## Bước 12: Chạy GRACE (llmpre.py)

Gửi đầy đủ: mã nguồn + prompt + graph (node + edge) + demonstration + label.
Đây là pipeline GRACE hoàn chỉnh đúng theo bài báo.

In [1]:
!python3 llmpre.py

/media/quang-dung/Windows-SSD/MY-FILES/LAB-THAY-THO/.venv-while-ubuntu/lib/python3.12/site-packages/vertexai/generative_models/_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()
Found existing results. Resuming from index 1417 / 2274
/media/quang-dung/Windows-SSD/MY-FILES/LAB-THAY-THO/.venv-while-ubuntu/lib/python3.12/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)
Raw: Looking at this code snippet, I need to analyze it for 